## IMPORTS + CONFIG

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from src.models.sugarcane_classifier import SugarcaneClassifier
from src.data.dataset import SugarcanDataset
from src.evaluation.evaluator import load_checkpoint, evaluate_on_test, print_report, get_confusion_matrix
from src.utils.config import load_config

## SETUP

In [ ]:
config = load_config("configs/obj1_cross_attention.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
class_names= config["data"]["class_names"]

test_ds = SugarcanDataset(split="test",config=config)
test_loader=DataLoader(test_ds,batch_size=config["data"]["batch_size"])

CHECKPOINT_PATH = "/content/drive/MyDrive/SugarcaneAI/checkpoints/best_model.pt"

## Load Model

In [ ]:
model = SugarcaneClassifier(num_classes=config["data"]["num_classes"]).to(device)
model = load_checkpoint(model,CHECKPOINT_PATH,device)

criterion= nn.CrossEntropyLoss()

## Evaluate

In [ ]:
avg_loss, metrics, all_preds, all_labels = evaluate_on_test(model,test_loader, criterion, device)

print(f"Test Loss : {avg_loss:.4f}")
print(f"Accuracy  : {metrics['accuracy']:.3f}")
print(f"F1 (macro): {metrics['f1_macro']:.3f}")
print()
print_report(all_preds, all_labels, class_names)

cm = get_confusion_matrix(all_preds, all_labels)
print("Confusion Matrix:")
print(cm)